## 0. Setup

Create required output directories. Run this once before anything else.

In [1]:
import os

os.makedirs("./pickles", exist_ok=True)
os.makedirs("./embeddings", exist_ok=True)
os.makedirs("./BERTopic", exist_ok=True)

# Verify the input file from step 00 exists
input_path = "./pickles/first-date_posts-all.pkl"
assert os.path.exists(input_path), (
    f"Input not found: {input_path}\n"
    "Run 00-dataset_preparation.ipynb first."
)
print("✅ Directories ready. Input file found.")

✅ Directories ready. Input file found.


## 1. Load Data

Load the preprocessed posts pickle and concatenate title + body into a single document string per post.

**To adjust:** Change the pickle path to point to your filtered dataset (e.g. swap `first-date_posts-all.pkl` for a different keyword-filtered file).

In [10]:
import pandas as pd

# Read the pickled DataFrame containing the posts
df = pd.read_pickle("./pickles/first-date_posts-all.pkl")

documents = df["title"] + ". " + df["selftext"].fillna("")
documents = documents.apply(lambda x: x.replace("\n", " "))
docs = documents.tolist()
docs

['Judging someone based on first date. Tell me what sticks out to you and if you’ve met anyone like this. I’m open to your best advice. F25 matched with a 30M on tinder. We hit it off immediately because we have very similar interests. From the start he is a yapper. Not open about the sensitive stuff but highlighted his reason for joining tinder, he’s looking for new social circle, and the existence of his kid. He told me about his job (humanitarian type) and asked equal questions about myself. We both liked a specific musician that was playing a show the following day so he asked me if I wanted to go with him, I agreed and we went together. He picked me up from my apt (I live in a shared apt). The first thing that sticks out is that he came to my gate instead of waiting for me in the car like most of my other dates. We went to the concert. Sat and talked about our likes, values, political views. We seemed to align with each other’s views since we engage in the same music culture. He h

## 2. Create Embeddings

Encode all documents using a SentenceTransformer model and cache the result to disk.

**To adjust:** Change `embedding_model_string` to use a different model (e.g. `paraphrase-multilingual-MiniLM-L12-v2` for non-English data). **Skip this cell** if embeddings are already cached — go straight to "Load Cached Embeddings" below.

In [11]:
# Create embeddings
from sentence_transformers import SentenceTransformer
import pickle

embedding_model_string = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(embedding_model_string, trust_remote_code=True)

embeddings = embedding_model.encode(docs, show_progress_bar=True)

with open(f"./embeddings/embeddings_updated_{embedding_model_string}.pkl", "wb") as f:
    pickle.dump(embeddings, f)

KeyboardInterrupt: 

## 3. Configure Sub-models

Define the four components that BERTopic composes internally:

| Model | Role | Key parameters |
|---|---|---|
| **UMAP** | Dimensionality reduction | `n_components`, `n_neighbors` (higher → more global structure), `min_dist` |
| **HDBSCAN** | Density-based clustering | `min_cluster_size` (higher → fewer, broader topics), `min_samples` |
| **c-TF-IDF** | Topic keyword representation | `reduce_frequent_words` removes corpus-wide stopwords |
| **CountVectorizer** | Vocabulary filtering | `min_df`/`max_df`, `ngram_range` (add `(1,3)` for trigrams) |

**To adjust:** Increase `min_cluster_size` to get fewer but cleaner topics. Decrease it for more granular topics. Change `ngram_range` to `(1, 3)` if topic keywords feel too generic.

In [3]:
from umap import UMAP

umap_model = UMAP(n_components=15, n_neighbors=65, min_dist=0.3, metric='cosine', random_state=42)
umap_model
from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(min_samples=10, min_cluster_size=90, metric='euclidean', cluster_selection_method='eom',
                        prediction_data=True)
hdbscan_model
from bertopic.vectorizers import ClassTfidfTransformer

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
ctfidf_model
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(stop_words="english", min_df=0.01, max_df=0.90, ngram_range=(1, 2))

/opt/homebrew/anaconda3/envs/RedditAnalysis/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4. Load Cached Embeddings

Load previously computed embeddings from disk. Run this cell **instead of** the "Create Embeddings" cell above when re-running the experiment to avoid re-encoding (which can take minutes to hours depending on dataset size).

**To adjust:** Make sure `embedding_model_string` matches the model used during encoding, since the file name encodes the model name.

In [12]:
# Load embeddings
import pickle

embedding_model_string = "all-MiniLM-L6-v2"
embeddings = pickle.load(open(f"./embeddings/embeddings_updated_{embedding_model_string}.pkl", "rb"))

## 5. Fit BERTopic

Instantiate BERTopic with the configured sub-models and fit it on the corpus using pre-computed embeddings.

- A pre-merge checkpoint is saved to `./BERTopic_pre-merge/` for reproducibility.
- The final merged model is saved to `./BERTopic/` in step 7 — this is what downstream notebooks load.
- `topic_info` (pre-merge) is saved to `./pickles/topic_info.pkl` for inspection.

**To adjust:** `top_n_words` sets how many representative keywords are stored per topic in the model output.

In [13]:
from bertopic import BERTopic

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    ctfidf_model=ctfidf_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    top_n_words=20,
    verbose=True)
topics, probs = topic_model.fit_transform(docs, embeddings)

# Save pre-merge checkpoint (for reproducibility / re-inspection without re-fitting)
topic_model.save("./BERTopic_pre-merge", serialization="safetensors", save_ctfidf=True,
                 save_embedding_model=embedding_model_string)

tmp = topic_model.get_document_info(docs)
tmp.to_pickle("./pickles/topic_info.pkl")

2026-04-19 14:21:23,521 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-19 14:22:58,149 - BERTopic - Dimensionality - Completed ✓
2026-04-19 14:22:58,151 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-19 14:23:09,354 - BERTopic - Cluster - Completed ✓
2026-04-19 14:23:09,360 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-19 14:23:26,035 - BERTopic - Representation - Completed ✓


## 6. Inspect Topics

Print the full topic info table (topic ID, size, top keywords). Use this output to decide which topics to merge in the next step.

> Topic `-1` is the outlier/noise bucket — documents that did not fit any cluster. A large `-1` topic suggests `min_cluster_size` may be too high.

In [14]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,57923,-1_morning_kept_texts_sent,"[morning, kept, texts, sent, replied, messages...",[First date went super well... not she (seems)...
1,0,8680,0_ideas_date ideas_tips_good date,"[ideas, date ideas, tips, good date, suggestio...",[First Date: How To Keep The Conversation Goin...
2,1,3196,1_texts_hes interested_respond_hed,"[texts, hes interested, respond, hed, havent h...",[Etiquette of rescheduling a second date?. Bac...
3,2,2872,2_matches_doing wrong_success_hobbies,"[matches, doing wrong, success, hobbies, ive t...",[Dating Roundtable : Ask you to read and share...
4,3,2044,3_text date_respond_texting date_texts,"[text date, respond, texting date, texts, text...","[Went on Four Great Dates, but Haven't Been ab..."
5,4,1870,4_paying_split_pay date_paid,"[paying, split, pay date, paid, money, offer, ...",[should i date him even if he has less money t...
6,5,1323,5_ghosted_ghosting_ghost_sent,"[ghosted, ghosting, ghost, sent, response, got...",[Ghosted because of Intimidation or Loss of At...
7,6,1148,6_ghosted_ghosting_ghost_got ghosted,"[ghosted, ghosting, ghost, got ghosted, mornin...",[Seriously cannot get over this guy ghosting m...
8,7,1026,7_flowers_gift_flowers date_valentines,"[flowers, gift, flowers date, valentines, vale...",[Should I get him a Valentine’s Day/birthday g...
9,8,677,8_wear_dress_wear date_jeans,"[wear, dress, wear date, jeans, outfit, wearin...","[First date attire?. Hey 👋, basically I’m goin..."


## 7. Manually Merge Topics

Merge semantically related topics into consolidated groups based on the inspection above.

Each inner list contains topic IDs to merge together. The first ID in each list becomes the target topic.

**To adjust:** After running "Inspect Topics", update `topics_to_merge` to reflect the topic groupings that make sense for your data. Uncomment any commented groups or add new ones. Topic IDs will differ if you re-ran BERTopic with different hyperparameters — always re-inspect before merging.

In [15]:
# Manually merge topics
topics_to_merge = [
    [5, 6, 16],  # ghosting
    [1, 3],  #responding
    [12],  #kids
    [10, 15, 21],  #kiss
    [9, 14],  #sex
    # [],  #age gap
    [18, 19],  #match on tinder
]
topic_model.merge_topics(docs, topics_to_merge)

# Save the model AFTER merging — this is what downstream notebooks (20-llm_labeling.ipynb) load
topic_model.save("./BERTopic", serialization="safetensors", save_ctfidf=True,
                 save_embedding_model=embedding_model_string)

## 8. Save Document-Topic Assignments

Export the final per-document topic assignments (after merging) to a pickle for use in downstream notebooks (`filtered-dating-advice_bertopic+LLM.ipynb`).

In [16]:
topic_model.get_document_info(docs).to_pickle("./pickles/topic_info_merged.pkl")

# BERTopic Visualizations

> ⚠️ **Prerequisites for cells below:**
> These visualization cells depend on outputs from **step ** (`filtered-dating-advice_bertopic+LLM.ipynb`). Run that notebook first and make sure the following files exist:
> - `./pickles/topic_metadata_llm.pkl` — LLM-generated topic labels
> - `./pickles/gemini-3-pro-classified_scripts_with_emotions.pkl` — classified scripts with emotion labels
>
> The cells also require `embeddings` to be in memory (run step 2 or 4 above in the same session).

In [9]:
import pandas as pd

# 1. Load your BERTopic model
# topic_model = BERTopic.load("...")

# 2. Load the metadata pickle
metadata_df = pd.read_pickle('./pickles/topic_metadata_llm.pkl')

# 3. Create the mapping dictionary
metadata_df['CustomLabel'] = metadata_df['CustomLabel'].str.replace('**', '', regex=False).str.strip()
labels_map = dict(zip(metadata_df['Topic'], metadata_df['CustomLabel']))

# 4. Apply to model
topic_model.set_topic_labels(labels_map)

# 5. (Optional) Merge descriptions into your topic info if needed for analysis
topic_info = topic_model.get_topic_info()
topic_info = topic_info.merge(metadata_df[['Topic', 'Description']], on='Topic', how='left')

FileNotFoundError: [Errno 2] No such file or directory: './pickles/topic_metadata_llm.pkl'

In [ ]:
import pandas as pd

# 1. Get the document info dataframe directly from BERTopic
df = topic_model.get_document_info(docs)

# 2. Load classified results with emotions
df_results = pd.read_pickle("./pickles/gemini-3-pro-classified_scripts_with_emotions.pkl")

# leave only relevant columns
df_results = df_results[[
    'selftext',
    'title',
    'predicted_script',
    'script_confidence',
    'Topic',
    'CustomLabel',
    'Description',
    'dominant_emotion']]

df_results['submission_text'] = (df_results["title"] + ". " + df_results["selftext"].fillna("")).astype(
    str).str.replace("\n", " ")

cols_to_merge = [
    'submission_text',
    'predicted_script',
    'script_confidence',
    'CustomLabel',
    'Description',
    'dominant_emotion'
]
actual_cols_to_merge = [col for col in cols_to_merge if col in df_results.columns]

df_results_clean = df_results.drop_duplicates(subset=['submission_text'])

df = df.merge(
    df_results_clean[actual_cols_to_merge],
    left_on='Document',
    right_on='submission_text',
    how='left'
)

missing_count = df['submission_text'].isna().sum()
print(f"Filling blanks for {missing_count} unclassified rows...")

df['dominant_emotion'] = df['dominant_emotion'].fillna('unclassified')
df['predicted_script'] = df['predicted_script'].fillna('No Script')
df['CustomLabel'] = df['CustomLabel'].fillna('Outlier / No Topic')

print(f"✅ Merge complete! Your dataframe has {len(df)} rows.")

# ==========================================
# GENERATE 2D UMAP COORDINATES
# ==========================================
# Reduce embeddings to 2D for scatter plot visualizations below.
# 'embeddings' must be in memory (run step 2 or 4 above in the same session).
from umap import UMAP

print("Reducing embeddings to 2D (this may take a minute)...")
umap_2d = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
reduced_embeddings = umap_2d.fit_transform(embeddings)
df['x'] = reduced_embeddings[:, 0]
df['y'] = reduced_embeddings[:, 1]
print("✅ X and Y coordinates added to dataframe.")

df_valid = df[df['Topic'] != -1]
topic_counts = df_valid['CustomName'].value_counts()
print("\n--- Top 5 Most Dominant Topics ---")
print(topic_counts.head(5))
print("\n--- Top 5 Least Dominant Topics ---")
print(topic_counts.tail(5).sort_values(ascending=True))
print(f"\nOutliers (Topic -1): {len(df[df['Topic'] == -1])}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patheffects as PathEffects
from adjustText import adjust_text

print("Configuring fonts...")
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['CMU Serif', 'DejaVu Serif', 'Times New Roman']

# SEPARATE OUTLIERS
outliers = df[df['Topic'] == -1]
non_outliers = df[df['Topic'] != -1]

# CALCULATE CLUSTER STATS (Centroids and Sizes)
cluster_stats = non_outliers.groupby('CustomName').agg(
    x=('x', 'median'),
    y=('y', 'median'),
    cluster_size=('Topic', 'count')
).reset_index()

# DYNAMIC FONT SCALING
min_size = cluster_stats['cluster_size'].min()
max_size = cluster_stats['cluster_size'].max()

MIN_FONT = 8
MAX_FONT = 18
cluster_stats['font_size'] = MIN_FONT + (MAX_FONT - MIN_FONT) * (
        (cluster_stats['cluster_size'] - min_size) / (max_size - min_size + 1e-5)
)

fig, ax = plt.subplots(figsize=(8, 10))

ax.scatter(
    outliers['x'], outliers['y'],
    color='#E0E0E0', s=2, alpha=0.3, edgecolors='none', zorder=0
)

unique_clusters = sorted(non_outliers['CustomName'].unique())
palette = sns.color_palette("husl", len(unique_clusters))
color_map = dict(zip(unique_clusters, palette))

sns.scatterplot(
    data=non_outliers, x='x', y='y', hue='CustomName',
    palette=color_map,
    s=3, alpha=0.5, edgecolors='none',
    legend=False, ax=ax, zorder=1
)

texts = []
for _, row in cluster_stats.iterrows():
    t = ax.text(
        row['x'], row['y'], row['CustomName'],
        fontsize=row['font_size'],
        weight='normal',
        color=color_map[row['CustomName']],
        ha='center', va='center', zorder=3
    )
    t.set_path_effects([PathEffects.withStroke(linewidth=3, foreground='white')])
    texts.append(t)

print("Adjusting text layout to prevent overlaps...")
adjust_text(
    texts,
    ax=ax,
    arrowprops=dict(arrowstyle='-', color='gray', alpha=0.5, lw=0.5),
    expand_points=(1.2, 1.2)
)

ax.axis('off')
plt.tight_layout()

output_path = "./pickles/Topic_Clusters_A4.png"
plt.savefig(output_path, format='png', bbox_inches='tight', dpi=300)
plt.show()
print(f"✅ Saved to: {output_path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patheffects as PathEffects
from adjustText import adjust_text
from scipy.spatial import cKDTree
from matplotlib.lines import Line2D

# ==========================================
# 0. THESIS FONT CONFIGURATION
# ==========================================
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Computer Modern Roman', 'CMU Serif', 'DejaVu Serif', 'Times New Roman']


def plot_dynamic_alpha_emotion_landscape(df):
    # ==========================================
    # 1. SEPARATE BACKGROUND & FOREGROUND
    # ==========================================
    background_mask = (df['Topic'] == -1) | (df['dominant_emotion'].isin(['neutral', 'unclassified']))
    background_df = df[background_mask]
    foreground_df = df[~background_mask].copy()

    # ==========================================
    # 2. EMOTION DENSITY CALCULATION (DOT OPACITY)
    # ==========================================
    radius_check = 0.1
    print("Calculating local density per emotion...")
    foreground_df['point_density'] = 0

    for emotion, group in foreground_df.groupby('dominant_emotion'):
        if len(group) < 2: continue
        tree = cKDTree(group[['x', 'y']])
        counts = tree.query_ball_tree(tree, r=radius_check)
        foreground_df.loc[group.index, 'point_density'] = [len(c) for c in counts]

    min_d = foreground_df['point_density'].min()
    max_d = foreground_df['point_density'].max()
    foreground_df['point_alpha'] = 0.15 + (0.85 - 0.15) * (
            (foreground_df['point_density'] - min_d) / (max_d - min_d + 1e-5)
    )

    # ==========================================
    # 3. TOPIC LABEL CALCULATION (ALL TOPICS)
    # ==========================================
    print("Calculating centroids and dynamic alphas for ALL topics...")
    valid_topics = df[df['Topic'] != -1]

    cluster_stats = valid_topics.groupby('CustomName').agg(
        x=('x', 'median'),
        y=('y', 'median'),
        cluster_size=('Topic', 'count')
    ).reset_index()

    min_size = cluster_stats['cluster_size'].min()
    max_size = cluster_stats['cluster_size'].max()

    # Calculate dynamic font size (6 Pt to 15 Pt to accommodate all topics)
    MIN_FONT, MAX_FONT = 6, 15
    cluster_stats['font_size'] = MIN_FONT + (MAX_FONT - MIN_FONT) * (
            (cluster_stats['cluster_size'] - min_size) / (max_size - min_size + 1e-5)
    )

    # --- KEY CHANGE: Calculate dynamic label alpha (0.35 to 0.95) ---
    # Smallest topics will be 35% opaque, largest will be 95% opaque
    MIN_ALPHA, MAX_ALPHA = 0.35, 0.95
    cluster_stats['label_alpha'] = MIN_ALPHA + (MAX_ALPHA - MIN_ALPHA) * (
            (cluster_stats['cluster_size'] - min_size) / (max_size - min_size + 1e-5)
    )

    # ==========================================
    # 4. PLOTTING
    # ==========================================
    fig, ax = plt.subplots(figsize=(8, 10))

    # Plot Background (Gray noise)
    ax.scatter(background_df['x'], background_df['y'], color='#E8E8E8', s=1, alpha=0.15, edgecolors='none', zorder=0)

    # Intuitive Emotion Colors
    emotion_colors = {
        'anger': '#e41a1c', 'sadness': '#377eb8', 'joy': '#ff7f00',
        'fear': '#984ea3', 'surprise': '#e6ab02', 'disgust': '#4daf4a'
    }

    # Plot Foreground Emotions
    for emotion, group in foreground_df.groupby('dominant_emotion'):
        color = emotion_colors.get(emotion, '#333333')
        ax.scatter(group['x'], group['y'], color=color, s=3.5, alpha=group['point_alpha'], edgecolors='none', zorder=1)

    # Plot All Topic Labels with matching alpha
    texts = []
    for _, row in cluster_stats.iterrows():
        t = ax.text(
            row['x'], row['y'], row['CustomName'],
            fontsize=row['font_size'],
            weight='normal',
            color='#222222',  # Very dark gray
            alpha=row['label_alpha'],  # --- NEW: Apply dynamic alpha here ---
            ha='center', va='center', zorder=3
        )
        # Apply the exact same alpha to the white stroke so it fades perfectly with the text
        t.set_path_effects([PathEffects.withStroke(linewidth=2.5, foreground='white', alpha=row['label_alpha'])])
        texts.append(t)

    # --- KEY CHANGE: Prevent overlap AND draw pointing lines ---
    print("Adjusting text layout and drawing connector lines (this takes time for 50+ labels)...")
    adjust_text(
        texts,
        ax=ax,
        expand_points=(2.0, 2.0),
        expand_objects=(1.5, 1.5),
        # arrowprops is what draws the line. 'arrowstyle=-' means no arrow head, just a line.
        arrowprops=dict(arrowstyle='-', color='#555555', alpha=0.6, lw=0.6, zorder=2)
    )

    # ==========================================
    # 5. LEGEND & EXPORT
    # ==========================================
    ax.axis('off')

    legend_elements = [Line2D([0], [0], marker='o', color='w', label=emo,
                              markerfacecolor=color, markersize=10)
                       for emo, color in emotion_colors.items() if emo in foreground_df['dominant_emotion'].unique()]

    ax.legend(handles=legend_elements, title="Dominant Emotion", loc="lower right",
              fontsize=11, title_fontsize=12, frameon=True, facecolor='white', edgecolor='black')

    plt.tight_layout()
    output_path = "./pickles/Thesis_DynamicLabel_EmotionMap.png"
    plt.savefig(output_path, format='png', bbox_inches='tight', dpi=300)
    plt.show()
    print(f"✅ Dynamic Label Emotion Map saved to: {output_path}")


# --- EXECUTION ---
plot_dynamic_alpha_emotion_landscape(df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def plot_top5_emotion_heatmap(df):
    valid_df = df[
        (df['Topic'] != -1) & (df['dominant_emotion'] != 'neutral') & (df['dominant_emotion'] != 'unclassified')].copy()

    if valid_df.empty:
        print("❌ No emotional data available.")
        return

    top_5_topics = valid_df['CustomLabel'].value_counts().nlargest(5).index
    plot_df = valid_df[valid_df['CustomLabel'].isin(top_5_topics)]

    heatmap_data = pd.crosstab(plot_df['CustomLabel'], plot_df['dominant_emotion'], normalize='index') * 100

    plt.figure(figsize=(10, 4))
    sns.set_theme(style="white")

    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Computer Modern Roman', 'CMU Serif', 'DejaVu Serif', 'Times New Roman']

    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".1f",
        cmap="Reds",
        cbar_kws={'label': 'Share of Emotional Posts (%)'},
        linewidths=.5,
        vmin=0, vmax=100
    )

    plt.title('Emotional Composition of Top 5 Dating Topics', fontsize=16, fontweight='bold', pad=15)
    plt.ylabel('Topic Cluster', fontsize=12, fontweight='bold')
    plt.xlabel('Expressed Emotion', fontsize=12, fontweight='bold')
    plt.xticks(fontsize=11)
    plt.yticks(fontsize=11, rotation=0)
    plt.tight_layout()

    output_path = "./pickles/Thesis_Top5_Emotions_300DPI.png"
    plt.savefig(output_path, format='png', bbox_inches='tight', dpi=300)
    plt.show()
    print(f"✅ Saved to: {output_path}")


plot_top5_emotion_heatmap(df)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patheffects as PathEffects
from adjustText import adjust_text

# ---------------------------------------------------------
# 1. SETUP COLORS & CENTROIDS
# ---------------------------------------------------------
# Filter out outliers (Topic -1) for the main color mapping
non_outliers = df[df['Topic'] != -1]

# Get unique labels
unique_labels = non_outliers['CustomName'].unique()

# Create a color palette with enough colors for all your topics
# 'husl' is good for generating many distinct colors
palette = sns.color_palette("husl", len(unique_labels))

# Map each label to a specific color so we can use it for both dots and text
label_color_map = dict(zip(unique_labels, palette))

# Calculate centroids using MEDIAN (often visually better than mean)
topic_centroids = non_outliers.groupby('CustomName')[['x', 'y']].median()

# ---------------------------------------------------------
# 2. CREATE THE PLOT
# ---------------------------------------------------------
plt.figure(figsize=(24, 15))  # Large size for clarity

# A. Plot Outliers (Topic -1) in the background
outliers = df[df['Topic'] == -1]
plt.scatter(
    outliers['x'], outliers['y'],
    color='#E0E0E0',
    s=10,
    alpha=0.3,
    linewidth=0,
    zorder=0  # Force to background
)

# B. Plot the Colored Clusters
# We iterate manually to ensure perfect color matching with our map
sns.scatterplot(
    data=non_outliers,
    x='x',
    y='y',
    hue='CustomName',
    palette=label_color_map,
    s=20,
    alpha=0.6,
    linewidth=0,
    legend=False,
    zorder=1
)

# ---------------------------------------------------------
# 3. ADD INTELLIGENT LABELS
# ---------------------------------------------------------
texts = []

# Loop through every topic centroid
for label, row in topic_centroids.iterrows():
    # Get the exact color assigned to this topic
    text_color = label_color_map[label]

    # Create the text object
    t = plt.text(
        row['x'],
        row['y'],
        label,
        fontdict={'weight': 'bold', 'size': 11},
        color=text_color,  # Text matches cluster color
        horizontalalignment='center',
        verticalalignment='center'
    )

    # Add a white 'halo' outline for readability
    # This acts like a background box but looks much cleaner
    t.set_path_effects([
        PathEffects.withStroke(linewidth=4, foreground='white')
    ])

    texts.append(t)

# ---------------------------------------------------------
# 4. OPTIMIZE LABEL POSITIONS (NON-OVERLAPPING)
# ---------------------------------------------------------
print("Adjusting text positions... (this may take a moment)")
adjust_text(
    texts,
    arrowprops=dict(arrowstyle='-', color='gray', alpha=0.5, lw=1),
    expand_points=(1.5, 1.5)  # Push text slightly away from dense dots
)

# ---------------------------------------------------------
# 5. FINALIZE
# ---------------------------------------------------------
plt.axis('off')
plt.tight_layout()
plt.show()